# 第 50 章：Capstone 采用反馈与维护节奏

这个 notebook 用一个极小的 adoption-feedback table，对比“只按采用影响排序”的 naive baseline 和“先检查严重度、责任人、证据与维护节奏”的课程维护 workflow。

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_adoption_feedback_maintenance_demo.csv')


def load_items(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['adoption_impact_score'] = float(row['adoption_impact_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_items(DATA_PATH)
print(f'Loaded {len(rows)} adoption-feedback items from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Feedback source:', ordered_counts(rows, 'feedback_source'))
print('Severity:', ordered_counts(rows, 'severity_status'))
print('Evidence:', ordered_counts(rows, 'evidence_status'))
print('Owner:', ordered_counts(rows, 'owner_status'))
print('Maintenance cadence:', ordered_counts(rows, 'maintenance_cadence_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['feedback_id']
    for row in rows
    if row['reference_route'] == 'log_for_next_release'
)
top4_impact = sorted(
    rows,
    key=lambda row: row['adoption_impact_score'],
    reverse=True,
)[:4]
baseline_ready_hits = sum(
    row['reference_route'] == 'log_for_next_release'
    for row in top4_impact
)
baseline_not_ready_hits = sum(
    row['reference_route'] != 'log_for_next_release'
    for row in top4_impact
)
baseline_ready_precision = baseline_ready_hits / len(top4_impact)
baseline_not_ready_intrusion = baseline_not_ready_hits / len(top4_impact)

print('Reference next-release items:', reference_ready_ids)
print('Top-4 by adoption impact:')
for row in top4_impact:
    print(
        f"  {row['feedback_id']}: impact={row['adoption_impact_score']:.2f}, "
        f"route={row['reference_route']}, area={row['affected_area']}"
    )
print('Baseline next-release precision:', round(baseline_ready_precision, 3))
print('Baseline not-ready intrusion:', round(baseline_not_ready_intrusion, 3))


In [ ]:
def route_feedback_item(row):
    if row['severity_status'] in {'critical', 'high'}:
        return 'triage_high_severity', 'critical or high-severity feedback needs immediate triage'
    if row['owner_status'] != 'assigned':
        return 'assign_maintenance_owner', 'feedback has no clear maintenance owner yet'
    if row['evidence_status'] != 'sufficient':
        return 'collect_more_evidence', 'feedback needs reproducible evidence before revision'
    if row['maintenance_cadence_status'] != 'scheduled':
        return 'schedule_maintenance_cadence', 'feedback needs a maintenance window or release cadence'
    return 'log_for_next_release', 'feedback has evidence, owner, bounded severity, and scheduled cadence'


routed_rows = []
for row in rows:
    route, reason = route_feedback_item(row)
    routed = dict(row)
    routed['route'] = route
    routed['reason'] = reason
    routed_rows.append(routed)

print('Maintenance workflow routes:')
for row in routed_rows:
    print(
        f"  {row['feedback_id']}: route={row['route']}, "
        f"reference={row['reference_route']}, reason={row['reason']}"
    )


In [ ]:
next_release_queue = [row for row in routed_rows if row['route'] == 'log_for_next_release']
evidence_queue = [row for row in routed_rows if row['route'] == 'collect_more_evidence']
severity_queue = [row for row in routed_rows if row['route'] == 'triage_high_severity']
owner_queue = [row for row in routed_rows if row['route'] == 'assign_maintenance_owner']
cadence_queue = [row for row in routed_rows if row['route'] == 'schedule_maintenance_cadence']

workflow_accuracy = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
) / len(routed_rows)
ready_precision = sum(
    row['reference_route'] == 'log_for_next_release'
    for row in next_release_queue
) / max(len(next_release_queue), 1)

print('Next-release queue:', [row['feedback_id'] for row in next_release_queue])
print('Collect-evidence queue:', [row['feedback_id'] for row in evidence_queue])
print('High-severity queue:', [row['feedback_id'] for row in severity_queue])
print('Assign-owner queue:', [row['feedback_id'] for row in owner_queue])
print('Schedule-cadence queue:', [row['feedback_id'] for row in cadence_queue])
print('Workflow route accuracy:', round(workflow_accuracy, 3))
print('Structured next-release precision:', round(ready_precision, 3))


In [ ]:
def maintenance_steps(row):
    steps = []
    if row['severity_status'] in {'critical', 'high'}:
        steps.append('先做高严重度 triage：确认影响范围、临时绕行方案和修复负责人。')
    if row['owner_status'] != 'assigned':
        steps.append('指定维护 owner，并记录判断、修订、复核和合并责任。')
    if row['evidence_status'] != 'sufficient':
        steps.append('补证据：复现步骤、截图、日志、环境和采用上下文。')
    if row['maintenance_cadence_status'] != 'scheduled':
        steps.append('安排维护节奏：检查日期、release window 和关闭规则。')
    return steps or ['可以记入下一版 release notes；保留反馈来源、版本和处理状态。']


for row in routed_rows:
    if row['route'] != 'log_for_next_release':
        print(f"\n{row['feedback_id']} -> {row['route']} ({row['affected_area']})")
        for step in maintenance_steps(row):
            print(' -', step)


In [ ]:
maintenance_cadence_outline = [
    'Feedback intake: issue, email, form, classroom note, or adoption interview',
    'Severity rubric: critical, high, medium, low, and defer criteria',
    'Evidence standard: reproduction steps, screenshots, environment, and context',
    'Owner path: triage owner, repair owner, reviewer, and release-note owner',
    'Maintenance windows: weekly hotfix, monthly cleanup, semester retrospective',
    'Closure rule: fixed, documented, deferred, duplicate, or out-of-scope',
]

feedback_assistant_prompt = '''你是我的 capstone 采用反馈与课程维护助手。

任务：
1. 先阅读 adoption-feedback table，不要只按 adoption impact 排序；
2. 先检查 severity；
3. 再检查 owner、evidence 和 maintenance cadence；
4. 把每条反馈 route 到 log_for_next_release、collect_more_evidence、
   triage_high_severity、assign_maintenance_owner 或 schedule_maintenance_cadence；
5. 对每条非 ready 反馈输出维护前 checklist。

输出格式：
- Next-release feedback
- Collect-evidence feedback
- High-severity feedback
- Assign-owner feedback
- Schedule-cadence feedback
- Maintenance checklist by feedback item
'''

print('Maintenance cadence outline:')
for item in maintenance_cadence_outline:
    print(' -', item)

print('\nAssistant prompt:')
print(feedback_assistant_prompt)


In [ ]:
maintenance_snapshot = {
    'dataset': DATA_PATH.name,
    'n_feedback_items': len(rows),
    'baseline_top4_next_release_precision': round(baseline_ready_precision, 3),
    'baseline_not_ready_intrusion': round(baseline_not_ready_intrusion, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'log_for_next_release': [row['feedback_id'] for row in next_release_queue],
    'collect_more_evidence': [row['feedback_id'] for row in evidence_queue],
    'triage_high_severity': [row['feedback_id'] for row in severity_queue],
    'assign_maintenance_owner': [row['feedback_id'] for row in owner_queue],
    'schedule_maintenance_cadence': [row['feedback_id'] for row in cadence_queue],
    'python_version': platform.python_version(),
}

print('Adoption feedback maintenance snapshot:')
for key, value in maintenance_snapshot.items():
    print(f'  {key}: {value}')
